In [5]:
# imports and config

import os, json, shutil, yaml
from tqdm import tqdm
from PIL import Image
from multiprocessing import Pool, cpu_count

DATA_ROOT = '/kaggle/input/datasets/varun000reddy/'
TRAIN_IMG = DATA_ROOT + 'training/train/image/'
TRAIN_ANN = DATA_ROOT + 'training/train/annos/'
VAL_IMG   = DATA_ROOT + 'validation/validation/image/'
VAL_ANN   = DATA_ROOT + 'validation/validation/annos/'

SAVE_ROOT = '/kaggle/working/yolo_dataset/'

# YOLO class mapping (0-based, no background)
TOP5 = {1: 0, 8: 1, 7: 2, 2: 3, 9: 4}
CLASS_NAMES = [
    'short_sleeve_top',
    'trousers',
    'shorts',
    'long_sleeve_top',
    'skirt'
]
NUM_CLASSES = len(CLASS_NAMES)

# Clean and recreate directory
shutil.rmtree(SAVE_ROOT, ignore_errors=True)
os.makedirs(SAVE_ROOT, exist_ok=True)

for split in ['train', 'val']:
    os.makedirs(f"{SAVE_ROOT}/images/{split}", exist_ok=True)
    os.makedirs(f"{SAVE_ROOT}/labels/{split}", exist_ok=True)

print(f"CPU cores available: {cpu_count()}")
print("Directory structure created")

CPU cores available: 4
Directory structure created


In [6]:
# Conversion Function

def process_single(args):
    img_name, img_dir, ann_dir, split = args
    img_id   = img_name.replace('.jpg', '')
    ann_path = os.path.join(ann_dir, img_id + '.json')

    if not os.path.exists(ann_path):
        return 'no_ann'

    try:
        with open(ann_path) as f:
            data = json.load(f)

        img_path = os.path.join(img_dir, img_name)
        img      = Image.open(img_path)
        w, h     = img.size

        yolo_lines = []

        for key, item in data.items():
            if not isinstance(item, dict):
                continue
            if not key.startswith('item'):
                continue

            cat = item.get('category_id')
            if cat not in TOP5:
                continue

            cls      = TOP5[cat]
            bbox     = item.get('bounding_box', [])
            segments = item.get('segmentation', [])

            if not bbox or not segments:
                continue

            # Use largest polygon as primary mask
            valid_segs = [s for s in segments if len(s) >= 6]
            if not valid_segs:
                continue

            largest_seg = max(valid_segs, key=len)
            coords = []
            for i in range(0, len(largest_seg), 2):
                x = max(0, min(largest_seg[i]   / w, 1.0))
                y = max(0, min(largest_seg[i+1] / h, 1.0))
                coords.append(f"{x:.6f}")
                coords.append(f"{y:.6f}")

            yolo_lines.append(f"{cls} " + " ".join(coords))

        # Skip images with no top-5 items — don't copy image either
        if not yolo_lines:
            return 'no_top5'

        # Save label file
        label_path = f"{SAVE_ROOT}/labels/{split}/{img_id}.txt"
        with open(label_path, 'w') as f:
            f.write("\n".join(yolo_lines))

        # Copy image
        shutil.copy(img_path,
                    f"{SAVE_ROOT}/images/{split}/{img_name}")

        return 'ok'

    except Exception as e:
        return f'error: {e}'


def convert_parallel(img_dir, ann_dir, split, max_samples=None):
    img_files = sorted([f for f in os.listdir(img_dir)
                        if f.endswith('.jpg')])

    if max_samples:
        img_files = img_files[:max_samples]

    print(f"\nConverting {split}: {len(img_files)} images "
          f"using {cpu_count()} CPU cores")

    args = [(img, img_dir, ann_dir, split) for img in img_files]

    results = {'ok': 0, 'no_top5': 0, 'no_ann': 0, 'error': 0}

    with Pool(cpu_count()) as pool:
        for r in tqdm(pool.imap(process_single, args),
                      total=len(args),
                      desc=f'Processing {split}'):
            if r == 'ok':
                results['ok'] += 1
            elif r == 'no_top5':
                results['no_top5'] += 1
            elif r == 'no_ann':
                results['no_ann'] += 1
            else:
                results['error'] += 1

    print(f"  Converted : {results['ok']}")
    print(f"  No top-5  : {results['no_top5']}")
    print(f"  No ann    : {results['no_ann']}")
    print(f"  Errors    : {results['error']}")
    return results['ok']

print("Conversion function ready")

Conversion function ready


In [7]:
# Convert train (70k samples) and val (full)

train_count = convert_parallel(TRAIN_IMG, TRAIN_ANN, 'train',
                               max_samples=70000)
val_count   = convert_parallel(VAL_IMG,   VAL_ANN,   'val',
                               max_samples=10000)  # limit val too

print(f"\nFinal counts:")
print(f"  Train images: {len(os.listdir(SAVE_ROOT + 'images/train'))}")
print(f"  Val images  : {len(os.listdir(SAVE_ROOT + 'images/val'))}")


Converting train: 70000 images using 4 CPU cores


Processing train: 100%|██████████| 70000/70000 [05:41<00:00, 205.26it/s]


  Converted : 52835
  No top-5  : 17165
  No ann    : 0
  Errors    : 0

Converting val: 10000 images using 4 CPU cores


Processing val: 100%|██████████| 10000/10000 [00:50<00:00, 196.82it/s]

  Converted : 6932
  No top-5  : 3068
  No ann    : 0
  Errors    : 0

Final counts:
  Train images: 52835
  Val images  : 6932


In [9]:
yaml_config = {
    'path' : '/kaggle/input/datasets/pankajdeopaiiitb/vr-yolo-dataset',
    'train': 'images/train',
    'val'  : 'images/val',
    'nc'   : NUM_CLASSES, 
    'names': CLASS_NAMES
}

yaml_path = f"{SAVE_ROOT}/data.yaml"
with open(yaml_path, 'w') as f:
    yaml.dump(yaml_config, f, default_flow_style=False)

print("data.yaml created:")
print(open(yaml_path).read())

data.yaml created:
names:
- short_sleeve_top
- trousers
- shorts
- long_sleeve_top
- skirt
nc: 5
path: /kaggle/input/datasets/pankajdeopaiiitb/vr-yolo-dataset
train: images/train
val: images/val



In [10]:
# Upload to Kaggle Dataset

import subprocess

metadata = {
    "title"   : "vr-yolo-dataset",
    "id"      : "pankajdeopaiiitb/vr-yolo-dataset",
    "licenses": [{"name": "CC0-1.0"}]
}
with open(f"{SAVE_ROOT}/dataset-metadata.json", 'w') as f:
    json.dump(metadata, f)

result = subprocess.run(
    ['kaggle', 'datasets', 'create', '-p', SAVE_ROOT, '--dir-mode', 'zip'],
    capture_output=True, text=True
)
print(result.stdout)
print(result.stderr)

Starting upload for file images.zip
Upload successful: images.zip (3GB)
Starting upload for file labels.zip
Upload successful: labels.zip (25MB)
Starting upload for file data.yaml
Upload successful: data.yaml (176B)
Your private Dataset is being created. Please check progress at https://www.kaggle.com/datasets/pankajdeopaiiitb/vr-yolo-dataset


  0%|          | 0.00/2.91G [00:00<?, ?B/s]
  1%|          | 19.5M/2.91G [00:00<00:15, 205MB/s]
  1%|▏         | 39.0M/2.91G [00:00<00:16, 185MB/s]
  2%|▏         | 57.3M/2.91G [00:00<00:16, 184MB/s]
  3%|▎         | 79.7M/2.91G [00:00<00:15, 199MB/s]
  3%|▎         | 102M/2.91G [00:00<00:14, 210MB/s] 
  4%|▍         | 122M/2.91G [00:00<00:15, 196MB/s]
  5%|▍         | 140M/2.91G [00:00<00:15, 188MB/s]
  5%|▌         | 159M/2.91G [00:00<00:16, 181MB/s]
  6%|▌         | 179M/2.91G [00:00<00:15, 190MB/s]
  7%|▋         | 197M/2.91G [00:01<00:15, 187MB/s]
  7%|▋         | 215M/2.91G [00:01<00:16, 181MB/s]
  8%|▊         | 232M/2.91G [00:01<00:16, 1